#A modular data sanitization and exploration engine for Google Colab.

End-to-end walkthrough of `DataInspector` + `PlottingMethods` on the **Titanic** dataset:
**Upload → Sanitize → Impute → Clean → Normalize → Visualize → Associations.**

> **Setup:** upload `data_inspector.py` to this Colab session (Files panel, or the upload cell below) before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#1. Dependencies
!pip -q install statsmodels plotly seaborn scipy scikit-learn
import plotly.io as pio
pio.renderers.default = "colab"

## Upload `data_inspector.py`
skip if file already added

In [ ]:
import os
if not os.path.exists('data_inspector.py'):
    from google.colab import files
    print('Upload data_inspector.py:')
    files.upload()
else:
    print('data_inspector.py found.')

data_inspector.py found.


In [ ]:
import os
if not os.path.exists('data_inspector.py'):
    from google.colab import files
    print('Upload data_inspector.py:')
    files.upload()
else:
    print('data_inspector.py found.')

data_inspector.py found.


## 1. Ingest data
`upload_data()` opens the Colab picker. For a reproducible demo we load Titanic from seaborn and inject some garbage strings (`?`, `n/a`) to exercise sanitization.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import seaborn as sns
from data_inspector import DataInspector, PlottingMethods

raw = sns.load_dataset('titanic')
raw['age'] = raw['age'].astype(object); raw['fare'] = raw['fare'].astype(object)
raw.loc[0:4, 'age'] = '?'
raw.loc[5:8, 'fare'] = 'n/a'

insp = DataInspector(raw)
insp.sanitize().auto_correct_types()
print('age ->', insp.df['age'].dtype, '| fare ->', insp.df['fare'].dtype)

age -> float64 | fare -> float64


## 2. Structural summary

In [ ]:
insp.data_summary(preview_rows=10)

DATASET SUMMARY
Rows: 891 | Columns: 15
Numeric Columns: 6
Categorical Columns: 9
--------------------------------------------------
Missing Values:
age            182
fare             4
embarked         2
deck           688
embark_town      2
--------------------------------------------------
Exact Duplicates: 107


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,NaN,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,NaN,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,NaN,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,NaN,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,NaN,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
5,0,3,male,NaN,0,0,NaN,Q,Third,man,True,NaN,Queenstown,no,True
6,0,1,male,54.0,0,0,NaN,S,First,man,True,E,Southampton,no,True
7,0,3,male,2.0,3,1,NaN,S,Third,child,False,NaN,Southampton,no,False
8,1,3,female,27.0,0,2,NaN,S,Third,woman,False,NaN,Southampton,yes,False
9,1,2,female,14.0,1,0,30.0708,C,Second,child,False,NaN,Cherbourg,yes,False


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,NaN,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,NaN,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,NaN,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,NaN,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,NaN,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
5,0,3,male,NaN,0,0,NaN,Q,Third,man,True,NaN,Queenstown,no,True
6,0,1,male,54.0,0,0,NaN,S,First,man,True,E,Southampton,no,True
7,0,3,male,2.0,3,1,NaN,S,Third,child,False,NaN,Southampton,no,False
8,1,3,female,27.0,0,2,NaN,S,Third,woman,False,NaN,Southampton,yes,False
9,1,2,female,14.0,1,0,30.0708,C,Second,child,False,NaN,Cherbourg,yes,False


## 3. Impute missing values
Median for skewed numerics, mode for categoricals.

In [ ]:
insp.handle_missing_values(strategy='median', columns='age,fare')
insp.handle_missing_values(strategy='mode', columns='embarked,embark_town,deck')
print('Remaining nulls:', int(insp.df.isna().sum().sum()))

Remaining nulls: 0


## 4. Clean: duplicates, outliers, targeted deletion

In [ ]:
insp.remove_duplicates()
insp.handle_outliers(columns='fare', action='flag')
insp.delete_columns('alive,is_outlier')
print('Shape now:', insp.df.shape)

## 5. Normalize (feature-engineering prep)

In [ ]:
num_scaled = insp.extract_normalized_numeric_data(method='standard', columns='age,fare')
cat_oh     = insp.extract_normalized_categorical_data(method='onehot', columns='sex,embarked')
model_ready = insp.merge_processed_data(numeric_method='minmax', categorical_method='ordinal')
print('scaled numeric:', num_scaled.shape, '| one-hot:', cat_oh.shape, '| merged:', model_ready.shape)
model_ready.head()

## 6. Visualize
### Univariate (3-panel) for a numeric column

In [ ]:
insp.plot_univariate('age').show()

### Smart relationships (auto-detected chart type)

In [ ]:
insp.plot_relationship('age', 'fare').show()
insp.plot_relationship('class', 'fare').show()
insp.plot_relationship('sex', 'class').show()

### Categorical frequency (counts + % labels)

In [ ]:
insp.plot_categorical_frequency('class').show()

## 7. Deep associations
Unified heatmap: |Pearson| (num-num), Cramér's V (cat-cat), Eta (mixed).

In [ ]:
insp.plot_all_associations_heatmap().show()

## 8. PlottingMethods — reusable HTML-wrapped charts
Returns embeddable HTML fragments; also renders inline here.

In [ ]:
from IPython.display import HTML
pm = PlottingMethods(insp.df)
HTML(pm.pie_chart(column='sex', title='Passengers by sex'))

In [ ]:
HTML(pm.bar_chart(column='class', title='Passengers by class'))

In [ ]:
HTML(pm.histogram(column='fare', bins=40, title='Fare distribution'))